In [1]:
%pip install -q ultralytics==8.4.215 roboflow

from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "Please select a T4 GPU runtime first"
print("GPU: ", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU:  Tesla T4


In [2]:
!pip install roboflow

from roboflow import Roboflow
import os
# Set ROBOFLOW_API_KEY in Colab Secrets or environment before running
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY"))
project = rf.workspace("sjp-gcfju").project("2026_greg")
version = project.version(4)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 2026_GreG-4 in yolov11:: 100%|██████████| 3964/3964 [00:00<00:00, 6255.70it/s]


In [3]:
from pathlib import Path
import yaml

data_dir = Path(dataset.location).resolve()
source_yaml = data_dir / "data.yaml"

with source_yaml.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

names = config["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda k: int(k))]

expected_names = [
    "Back", "Front", "left_side",
    "right_side", "upright", "upside_down"
]
assert names == expected_names, f"Class names or order changed; please check: {names}"

config["path"] = str(data_dir)
config["names"] = names
config["nc"] = len(names)

for split, folder in [
    ("train", "train"),
    ("val", "valid"),
    ("test", "test"),
]:
    image_dir = data_dir / folder / "images"
    assert image_dir.is_dir(), f"Image directory not found: {image_dir}"

    count = sum(
        p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        for p in image_dir.iterdir() if p.is_file()
    )
    assert count > 0, f"{split} has no images"
    config[split] = str(image_dir)
    print(f"{split}: {count} images")

data_yaml = data_dir / "data_colab.yaml"
with data_yaml.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

print("Dataset root: ", data_dir)
print("Classes: ", names)
print("Training config: ", data_yaml)

train: 1932 images
val: 26 images
test: 22 images
Dataset root:  /content/2026_GreG-4
Classes:  ['Back', 'Front', 'left_side', 'right_side', 'upright', 'upside_down']
Training config:  /content/2026_GreG-4/data_colab.yaml


In [4]:
from pathlib import Path
from ultralytics import YOLO
from datetime import datetime
from zoneinfo import ZoneInfo
import torch

assert torch.cuda.is_available(), "GPU not connected; please select a GPU runtime"
assert Path(data_yaml).is_file(), "Please run cell 3 first"
assert Path("/content/drive/MyDrive").is_dir(), "Please mount Google Drive first"

run_name = "paper_cups_640_full120_" + datetime.now(
    ZoneInfo("Europe/Stockholm")
).strftime("%Y%m%d_%H%M%S")

# Same as round 3: start from official pretrained weights
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(data_yaml),
    imgsz=640,
    epochs=120,
    batch=8,
    device=0,
    workers=2,
    patience=0,       # Disable early stopping and train full 120 epochs
    seed=42,
    optimizer="auto",

    fliplr=0.0,
    flipud=0.0,
    degrees=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    plots=True,
    save=True,
    project="/content/drive/MyDrive/paper_cup_yolo11",
    name=run_name,
    exist_ok=False,
)

run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"

print("Run directory: ", run_dir)
print("Best weights: ", best_pt)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
New https://pypi.org/project/ultralytics/8.4.216 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/2026_GreG-4/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, em